# [Lab 1](https://github.com/hanggrian/IIT-CS587/blob/assets/assignments/lab1.pdf): AutoGen Introduction

The host machine for all labs is an EndeavourOS with Arch Linux as the base system.

## Requirements

1.  [x] Install Python version 3.10: [python310<sup>AUR</sup>](https://aur.archlinux.org/packages/python310)
1.  [x] Install VSCode: [visual-studio-code-bin<sup>AUR</sup>](https://aur.archlinux.org/packages/visual-studio-code-bin)
1.  [x] Check Python version.
    ```sh
    python --version
    ```
1.  [x] Create a virtual environment and link notebook kernel.
    ```sh
    source .venv/bin/activate
    python -m ipykernel install --user --name=.venv
    ```
1.  [x] Install packages:
    - `uv` project manager.
      ```sh
      pip install uv
      ```
    - `pyautogen` and dependencies for lessons.
      ```
      uv pip install -r requirements.txt
      ```
1.  [x] Create an OpenAI account and API key.
1.  [x] Set up the environment variable.
    ```sh
    echo 'OPENAI_API_KEY=XXXX-XXXX' >> ~/.env
    ```
1.  [x] Create a Deep Learning account and complete **AI Agentic Design Patterns with AutoGen:**
    - [Video introduction](https://learn.deeplearning.ai/courses/ai-agentic-design-patterns-with-autogen/lesson/1/introduction)
    - [Example 6: Planning and Stock Report Generation](https://learn.deeplearning.ai/courses/ai-agentic-design-patterns-with-autogen/lesson/7/planning-and-stock-report-generation)

## Setup

In [42]:
from utils import get_openai_api_key

llm_config = {
    'model': 'gpt-4-turbo',
    'api_key': get_openai_api_key(),
}

task = \
    'Write a blogpost about the stock price performance of ' + \
    "Nvidia in the past month. Today's date is 2024-04-23."

## Build a group chat

> This group chat will include these agents:
>
> 1.  **User_proxy or Admin:** to allow the user to comment on the report and ask the writer to refine it.
> 1.  **Planner:** to determine relevant information needed to complete the task.
> 1.  **Engineer:** to write code using the defined plan by the planner.
> 1.  **Executor:** to execute the code written by the engineer.
> 1.  **Writer:** to write the report.
>
> **Note:** In this lesson, you'll use an alternative method of code execution by providing a dict config. However, you can always use the LocalCommandLineCodeExecutor if you prefer. For more details about code_execution_config, check [this](https://microsoft.github.io/autogen/docs/reference/agentchat/conversable_agent/#__init__).

In [43]:
from autogen import ConversableAgent, AssistantAgent

user_proxy = \
    ConversableAgent(
        name='Admin',
        system_message='Give the task, and send ' +
            'instructions to writer to refine the blog post.',
        code_execution_config=False,
        llm_config=llm_config,
        human_input_mode='ALWAYS',
    )
planner = \
    ConversableAgent(
        name='Planner',
        system_message='Given a task, please determine '
            'what information is needed to complete the task. '
            'Please note that the information will all be retrieved using'
            ' Python code. Please only suggest information that can be '
            'retrieved using Python code. '
            'After each step is done by others, check the progress and '
            'instruct the remaining steps. If a step fails, try to '
            'workaround',
        description='Planner. Given a task, determine what '
            'information is needed to complete the task. '
            'After each step is done by others, check the progress and '
            'instruct the remaining steps',
        llm_config=llm_config,
    )
engineer = \
    AssistantAgent(
        name='Engineer',
        llm_config=llm_config,
        description='An engineer that writes code based on the plan '
            'provided by the planner.',
    )
executor = \
    ConversableAgent(
        name='Executor',
        system_message='Execute the code written by the '
            'engineer and report the result.',
        human_input_mode='NEVER',
        code_execution_config={
            'last_n_messages': 3,
            'work_dir': 'coding',
            'use_docker': False,
        },
    )
writer = \
    ConversableAgent(
        name='Writer',
        llm_config=llm_config,
        system_message='Writer.'
            'Please write blogs in markdown format (with relevant titles)'
            ' and put the content in pseudo ```md``` code block. '
            'You take feedback from the admin and refine your blog.',
        description='Writer.'
            'Write blogs based on the code execution results and take '
            'feedback from the admin to refine the blog.',
    )

## Start the group chat

> **Note:** In this lesson, you will use GPT 4 for better results. Please note that the lesson has a quota limit. If you want to explore the code in this lesson further, we recommend trying it locally with your own API key.

In [48]:
from autogen import GroupChat, GroupChatManager

groupchat = \
    GroupChat(
        agents=[user_proxy, engineer, writer, executor, planner],
        messages=[],
        max_round=10,
    )
manager = \
    GroupChatManager(
        groupchat=groupchat, 
        llm_config=llm_config,
    )

groupchat_result = user_proxy.initiate_chat(manager, message=task)

Admin (to chat_manager):

Write a blogpost about the stock price performance of Nvidia in the past month. Today's date is 2024-04-23.

--------------------------------------------------------------------------------

Next speaker: Planner


>>>>>>>> USING AUTO REPLY...
Planner (to chat_manager):

To write the blog post about Nvidia's stock price performance over the past month using Python, we need to gather and analyze the relevant data. Here’s a step-by-step breakdown of the information required:

1. Collect Nvidia's daily stock price data for the past month (from 2024-03-23 to 2024-04-23).
2. Analyze this data to determine key statistics such as the high, low, average, and closing prices. Also, determine any significant changes or trends.
3. Compare Nvidia's stock performance with general market trends or benchmarks (like S&P 500) during the same period.

These tasks involve fetching and processing financial data, which can be performed using Python libraries such as `pandas_datarea

## Add a speaker selection policy

In [49]:
user_proxy = \
    ConversableAgent(
        name='Admin',
        system_message='Give the task, and send '
            'instructions to writer to refine the blog post.',
        code_execution_config=False,
        llm_config=llm_config,
        human_input_mode='ALWAYS',
    )
planner = \
    ConversableAgent(
        name='Planner',
        system_message='Given a task, please determine '
            'what information is needed to complete the task. '
            'Please note that the information will all be retrieved using'
            ' Python code. Please only suggest information that can be '
            'retrieved using Python code. '
            'After each step is done by others, check the progress and '
            'instruct the remaining steps. If a step fails, try to '
            'workaround',
        description='Given a task, determine what '
            'information is needed to complete the task. '
            'After each step is done by others, check the progress and '
            'instruct the remaining steps',
        llm_config=llm_config,
    )
engineer = \
    AssistantAgent(
        name='Engineer',
        llm_config=llm_config,
        description='Write code based on the plan '
            'provided by the planner.',
    )
writer = \
    ConversableAgent(
        name='Writer',
        llm_config=llm_config,
        system_message='Writer. '
            'Please write blogs in markdown format (with relevant titles)'
            ' and put the content in pseudo ```md``` code block. '
            'You take feedback from the admin and refine your blog.',
        description='After all the info is available, '
            'write blogs based on the code execution results and take '
            'feedback from the admin to refine the blog. ',
    )
executor = \
    ConversableAgent(
        name='Executor',
        description='Execute the code written by the '
            'engineer and report the result.',
        human_input_mode='NEVER',
        code_execution_config={
            'last_n_messages': 3,
            'work_dir': 'coding',
            'use_docker': False,
        },
    )

In [50]:
groupchat = \
    GroupChat(
        agents=[user_proxy, engineer, writer, executor, planner],
        messages=[],
        max_round=10,
        allowed_or_disallowed_speaker_transitions={
            user_proxy: [engineer, writer, executor, planner],
            engineer: [user_proxy, executor],
            writer: [user_proxy, planner],
            executor: [user_proxy, engineer, planner],
            planner: [user_proxy, engineer, writer],
        },
        speaker_transitions_type='allowed',
    )

manager = GroupChatManager(groupchat=groupchat, llm_config=llm_config)
groupchat_result = user_proxy.initiate_chat(manager, message=task)

Admin (to chat_manager):

Write a blogpost about the stock price performance of Nvidia in the past month. Today's date is 2024-04-23.

--------------------------------------------------------------------------------

Next speaker: Planner


>>>>>>>> USING AUTO REPLY...
Planner (to chat_manager):

To write the blog post about Nvidia's stock price performance over the past month using Python, we need to gather and analyze the relevant data. Here’s a step-by-step breakdown of the information required:

1. Collect Nvidia's daily stock price data for the past month (from 2024-03-23 to 2024-04-23).
2. Analyze this data to determine key statistics such as the high, low, average, and closing prices. Also, determine any significant changes or trends.
3. Compare Nvidia's stock performance with general market trends or benchmarks (like S&P 500) during the same period.

These tasks involve fetching and processing financial data, which can be performed using Python libraries such as `pandas_datarea